# Review 2 — Bagging Classifier

**Track:** Classification Part B  
**Dataset:** Bank Marketing (`bank-full.csv`)  
**Target:** `y` — no/yes  

This is a standalone notebook with separate audit, preprocessing, training, evaluation, confusion matrix, tuning and model-saving cells.

## 1. Imports and dataset audit

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import joblib
ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'bank-full.csv').is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Dataset not found: data/bank-full.csv')
    ROOT = ROOT.parent
DATA=ROOT/'data'/'bank-full.csv'; RESULTS=ROOT/'results'/'classification'; FIGURES=ROOT/'figures'/'classification'; MODELS=ROOT/'models'/'classification'
for p in [RESULTS,FIGURES,MODELS]: p.mkdir(parents=True,exist_ok=True)
RANDOM_STATE=42
df=pd.read_csv(DATA,sep=';')
for col in df.select_dtypes(include='object').columns: df[col]=df[col].astype(str).str.strip().replace({'unknown':np.nan,'?':np.nan,'nan':np.nan})
y=df.pop('y').map({'no':0,'yes':1}); valid=y.notna(); X=df.loc[valid].copy(); y=y.loc[valid].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=RANDOM_STATE,stratify=y)
print('Dataset shape:',df.shape,'Train:',X_train.shape,'Test:',X_test.shape)
display(df.head()); display(df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))
def preprocessor(X, scale=False):
 num=X.select_dtypes(include=np.number).columns.tolist(); cat=X.select_dtypes(exclude=np.number).columns.tolist()
 ns=[('imputer',SimpleImputer(strategy='median'))]
 if scale: ns.append(('scaler',StandardScaler()))
 cs=[('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]
 return ColumnTransformer([('num',Pipeline(ns),num),('cat',Pipeline(cs),cat)])

## 2. Define and fit the model pipeline

In [ ]:
ALGORITHM='Bagging (Decision Tree)'
model=BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),n_estimators=50,random_state=RANDOM_STATE,n_jobs=-1)
pipe=Pipeline([('preprocess',preprocessor(X_train,scale=False)),('model',model)])
print(pipe)

## 3. Train the model

In [ ]:
pipe.fit(X_train,y_train)
print('Training completed.')

## 4. Evaluate on the held-out test set

In [ ]:
pred=pipe.predict(X_test)
score=pipe.predict_proba(X_test)[:,1] if hasattr(pipe,'predict_proba') else pipe.decision_function(X_test)
metrics=pd.DataFrame([{'algorithm':ALGORITHM,'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'weighted_f1':f1_score(y_test,pred,average='weighted',zero_division=0),'roc_auc':roc_auc_score(y_test,score)}])
display(metrics.round(4))
metrics.to_csv(RESULTS/'09_bagging_classifier.csv',index=False)

## 5. Confusion matrix

In [ ]:
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['No','Yes'],cmap='Blues',ax=ax)
ax.set_title('Bagging Classifier — Confusion Matrix');fig.tight_layout();fig.savefig(FIGURES/'09_bagging_classifier_confusion_matrix.png',dpi=150);plt.show()

## 6. Hyperparameter tuning (GridSearchCV)

In [ ]:
grid={'model__n_estimators':[25,50,100]}
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
search=GridSearchCV(pipe,grid,scoring='f1_weighted',cv=cv,n_jobs=-1,refit=True)
search.fit(X_train,y_train)
print('Best parameters:',search.best_params_)
print('Best CV weighted F1:',round(search.best_score_,4))
tuned=search.best_estimator_.predict(X_test)
tuned_score=search.best_estimator_.predict_proba(X_test)[:,1] if hasattr(search.best_estimator_,'predict_proba') else search.best_estimator_.decision_function(X_test)
tuned_metrics={'accuracy':accuracy_score(y_test,tuned),'precision':precision_score(y_test,tuned,zero_division=0),'recall':recall_score(y_test,tuned,zero_division=0),'weighted_f1':f1_score(y_test,tuned,average='weighted',zero_division=0),'roc_auc':roc_auc_score(y_test,tuned_score)}
display(pd.DataFrame([{'version':'baseline',**metrics.iloc[0].to_dict()},{'version':'tuned',**tuned_metrics}]))
pd.DataFrame([{'best_params':str(search.best_params_),'best_cv_weighted_f1':search.best_score_,**tuned_metrics}]).to_csv(RESULTS/'09_bagging_classifier_tuning.csv',index=False)

## 7. Save model and print report

In [ ]:
joblib.dump(search.best_estimator_,MODELS/'09_bagging_classifier.joblib')
print(classification_report(y_test,tuned,target_names=['No','Yes'],zero_division=0))